# Прогноз спроса на прокат велосипедов
**Датасет:** UCI Capital Bikeshare 2011–2012 (`hour.csv`)  
**Цель:** предсказать количество арендованных велосипедов за час (`cnt`)

## 1. Загрузка данных

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pickle
import json

df = pd.read_csv('../data/hour.csv')
print(f'Всего записей: {len(df)}')
df.head()

## 2. Признаки и целевая переменная

| Признак | Описание |
|---|---|
| `season` | Сезон (1–4) |
| `yr` | Год (0=2011, 1=2012) |
| `mnth` | Месяц (1–12) |
| `hr` | Час суток (0–23) — сильнейший предиктор |
| `holiday` | Праздник (0/1) |
| `weekday` | День недели (0–6) |
| `workingday` | Рабочий день (0/1) |
| `weathersit` | Погодные условия (1–4) |
| `temp` | Нормализованная температура |
| `atemp` | Ощущаемая температура |
| `hum` | Влажность |
| `windspeed` | Скорость ветра |

**Исключаем:** `instant` (порядковый номер), `dteday` (дублируется yr/mnth/hr), `casual`/`registered` (части суммы `cnt` — data leakage).

In [ ]:
FEATURES = [
    'season', 'yr', 'mnth', 'hr',
    'holiday', 'weekday', 'workingday',
    'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
]
TARGET = 'cnt'

X = df[FEATURES].copy()
y = df[TARGET].copy()

print('Распределение целевой переменной cnt:')
y.describe()

## 3. Кодирование категориальных признаков

`season` и `weathersit` — порядковые категории без числового смысла.  
Применяем **one-hot encoding**, чтобы модель не считала, что `winter (4)` в два раза «больше», чем `summer (2)`.

In [ ]:
season_map  = {1: 'spring', 2: 'summer', 3: 'fall', 4: 'winter'}
weather_map = {1: 'clear', 2: 'mist', 3: 'light_rain', 4: 'heavy_rain'}

X['season']     = X['season'].map(season_map)
X['weathersit'] = X['weathersit'].map(weather_map)

X = pd.get_dummies(X, columns=['season', 'weathersit'], drop_first=False)

X['holiday']    = X['holiday'].astype(int)
X['workingday'] = X['workingday'].astype(int)

print(f'Признаков после one-hot: {len(X.columns)}')
print(list(X.columns))

## 4. Масштабирование (Z-score)

Для непрерывных признаков применяем стандартизацию:
$$z = \frac{x - \mu}{\sigma}$$

Для RandomForest это не обязательно (деревья нечувствительны к масштабу), но **параметры μ и σ нужно сохранить**, чтобы `app.py` применял ту же трансформацию к входным данным пользователя.

In [ ]:
CONTINUOUS = ['temp', 'atemp', 'hum', 'windspeed']

means = X[CONTINUOUS].mean()
stds  = X[CONTINUOUS].std()

X[CONTINUOUS] = (X[CONTINUOUS] - means) / stds

print('Средние значения:')
print(means.round(4))
print('\nСтандартные отклонения:')
print(stds.round(4))

## 5. Разделение на train / test (80 / 20)

Используем **хронологическое разделение** — первые 80% записей в train, последние 20% в test.  
Случайное перемешивание здесь нельзя: данные временны́е, и случайное разбиение подмешало бы «будущее» в обучение.

In [ ]:
split_idx = int(len(X) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Всего записей : {len(df)}')
print(f'Train size    : {len(X_train)}  ({len(X_train)/len(df)*100:.0f}%)')
print(f'Test size     : {len(X_test)}   ({len(X_test)/len(df)*100:.0f}%)')

## 6. Кросс-валидация

Используем `TimeSeriesSplit` — он делит данные так, что обучение **всегда предшествует** валидации.  
Обычный `KFold` здесь некорректен: он случайно подмешивает «будущее» в обучение, что даёт завышенную оценку.

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred):
    mask = np.array(y_true) > 0
    return np.mean(np.abs(
        (np.array(y_true)[mask] - np.array(y_pred)[mask]) / np.array(y_true)[mask]
    )) * 100

lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

tscv = TimeSeriesSplit(n_splits=5)
cv_mae = {'LinearRegression': [], 'RandomForest': []}

print('--- Кросс-валидация (5 фолдов, TimeSeriesSplit) ---')
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), 1):
    Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    ytr, yval = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    lr.fit(Xtr, ytr)
    cv_mae['LinearRegression'].append(mean_absolute_error(yval, lr.predict(Xval)))

    rf.fit(Xtr, ytr)
    cv_mae['RandomForest'].append(mean_absolute_error(yval, rf.predict(Xval)))

    print(f'  Fold {fold}: LR MAE={cv_mae["LinearRegression"][-1]:.1f}  RF MAE={cv_mae["RandomForest"][-1]:.1f}')

print('\n--- Средняя MAE по кросс-валидации ---')
for name, scores in cv_mae.items():
    print(f'  {name:<20} {np.mean(scores):.1f} +/- {np.std(scores):.1f}')

## 7. Финальное обучение и метрики на тесте

In [ ]:
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)

p1_test = lr.predict(X_test)
p2_test = rf.predict(X_test)

print('--- Метрики на тестовой выборке ---')
for name, pred in [('LinearRegression', p1_test), ('RandomForest', p2_test)]:
    print(f'  {name:<20} MAE={mean_absolute_error(y_test, pred):6.1f}  '
          f'RMSE={rmse(y_test, pred):6.1f}  MAPE={mape(y_test, pred):5.1f}%')

## 8. Диагностика: MAE по часу суток

In [ ]:
hours  = X_test['hr'].values
err_lr = np.abs(y_test.values - p1_test)
err_rf = np.abs(y_test.values - p2_test)

mae_h_lr = pd.Series(err_lr).groupby(hours).mean()
mae_h_rf = pd.Series(err_rf).groupby(hours).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x, w = np.arange(24), 0.35
axes[0].bar(x - w/2, mae_h_lr, w, label='LinearRegression', color='steelblue')
axes[0].bar(x + w/2, mae_h_rf, w, label='RandomForest',     color='tomato')
axes[0].set_xlabel('Час суток')
axes[0].set_ylabel('MAE (велосипедов)')
axes[0].set_title('Где модель ошибается больше всего?')
axes[0].set_xticks(x)
axes[0].legend()

axes[1].scatter(y_test, p2_test, alpha=0.15, s=6, color='tomato')
lim = max(y_test.max(), p2_test.max()) + 10
axes[1].plot([0, lim], [0, lim], 'k--', linewidth=1, label='Идеальное предсказание')
axes[1].set_xlabel('Реальное cnt')
axes[1].set_ylabel('Предсказанное cnt')
axes[1].set_title('RandomForest: реальные vs предсказанные')
axes[1].legend()

plt.tight_layout()
plt.savefig('../diagnostics.png', dpi=150)
plt.show()
print('График сохранён: diagnostics.png')

## 9. Сохранение модели и метаданных

Сохраняем модель в `model.pkl` и параметры Z-score в `model_meta.json` — они нужны `app.py` для корректной трансформации входных данных пользователя.

In [ ]:
best_model = rf

final_pred = best_model.predict(X_test)
final_mae  = mean_absolute_error(y_test, final_pred)
final_rmse = rmse(y_test, final_pred)
final_mape = mape(y_test, final_pred)

err_by_hour = pd.Series(np.abs(y_test.values - final_pred), index=X_test['hr'].values)
worst_hours = err_by_hour.groupby(level=0).mean().nlargest(2).index.tolist()
mae_by_hour = {str(int(h)): round(v, 1)
               for h, v in err_by_hour.groupby(level=0).mean().items()}

with open('../model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

meta = {
    'model':    'RandomForest',
    'features': list(X_train.columns),
    'metrics':  {'MAE': round(final_mae, 2), 'RMSE': round(final_rmse, 2), 'MAPE': round(final_mape, 2)},
    'worst_hours': worst_hours,
    'mae_by_hour': mae_by_hour,
    'scaler': {'means': means.to_dict(), 'stds': stds.to_dict()},
}
with open('../model_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('=' * 52)
print('  ИТОГОВЫЙ ОТЧЁТ')
print('=' * 52)
print(f'  MAE  на тесте    : {final_mae:.1f}  велосипедов/час')
print(f'  RMSE на тесте    : {final_rmse:.1f}')
print(f'  MAPE на тесте    : {final_mape:.1f}%')
print(f'  Чаще всего путает: часы {worst_hours[0]}:00 и {worst_hours[1]}:00')
print(f'  Модель сохранена : model.pkl')
print(f'  Метаданные       : model_meta.json')
print('=' * 52)